In [1]:
import os, glob, re, random
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

FAKE_DIR = "fake_images"
N_FAKE_MAX = 1000
N_REAL_MAX = 1000
IMG_SIZE = 224
BATCH_SIZE = 64

real_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

fake_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# REAL: STL10
real_ds_full = torchvision.datasets.STL10(
    root="./data", split="train", download=True, transform=real_transform
)
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(real_ds_full), generator=g).tolist()
real_indices = perm[:N_REAL_MAX]
real_subset = torch.utils.data.Subset(real_ds_full, real_indices)

class RealWithMeta(Dataset):
    def __init__(self, subset): self.subset = subset
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        x, _ = self.subset[idx]
        return {"x": x, "y": torch.tensor(0), "path": None, "prompt_id": -1, "is_fake": False}

# FAKE: from disk with prompt_id parsing p{pid}_{i}.png
fake_paths = sorted(glob.glob(os.path.join(FAKE_DIR, "*.png")))[:N_FAKE_MAX]
assert len(fake_paths) > 0, f"No fake images in {FAKE_DIR}"
_prompt_re = re.compile(r".*[/\\]p(\d+)_\d+\.png$")

class FakeFolderWithMeta(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        x = self.transform(img)
        m = _prompt_re.match(p)
        pid = int(m.group(1)) if m else -1
        return {"x": x, "y": torch.tensor(1), "path": p, "prompt_id": pid, "is_fake": True}

class ConcatMeta(Dataset):
    def __init__(self, a, b): self.a, self.b = a, b
    def __len__(self): return len(self.a) + len(self.b)
    def __getitem__(self, idx):
        return self.a[idx] if idx < len(self.a) else self.b[idx - len(self.a)]

def collate_meta(batch):
    x = torch.stack([b["x"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0).long()
    pid = torch.tensor([b["prompt_id"] for b in batch], dtype=torch.long)
    is_fake = torch.tensor([int(b["is_fake"]) for b in batch], dtype=torch.long)
    paths = [b["path"] for b in batch]
    return x, y, pid, is_fake, paths

real_meta = RealWithMeta(real_subset)
fake_meta = FakeFolderWithMeta(fake_paths, fake_transform)
full_meta = ConcatMeta(real_meta, fake_meta)

n_train = int(0.8 * len(full_meta))
n_test  = len(full_meta) - n_train
train_meta, test_meta = random_split(full_meta, [n_train, n_test], generator=torch.Generator().manual_seed(0))

train_loader_meta = DataLoader(train_meta, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_meta)
test_loader_meta  = DataLoader(test_meta,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_meta)

print("train:", len(train_meta), "test:", len(test_meta))

device: cuda
train: 1600 test: 400


In [2]:
!pip -q install -U transformers
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPVisionModel

class ClipDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        hidden = self.backbone.config.hidden_size
        self.head = nn.Linear(hidden, 2)

    def forward(self, x01):
        # x01 in [0,1]
        mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x01.device).view(1,3,1,1)
        std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x01.device).view(1,3,1,1)
        x = (x01 - mean) / std
        feats = self.backbone(pixel_values=x).pooler_output
        return self.head(feats)

model = ClipDetector().to(device)

# Freeze backbone (as before)
for p in model.backbone.parameters():
    p.requires_grad = False

CLIP_CKPT = os.path.join("paper_outputs", "clip_detector.pt")  # if you saved it
if os.path.exists(CLIP_CKPT):
    model.load_state_dict(torch.load(CLIP_CKPT, map_location=device))
    print("Loaded CLIP detector weights:", CLIP_CKPT)
else:
    print("No clip_detector.pt found. Will quick-train head for 1 epoch.")
    opt = torch.optim.AdamW(model.head.parameters(), lr=3e-4)
    from tqdm.auto import tqdm
    model.train()
    for x, y, *_ in tqdm(train_loader_meta, desc="quick-train-clip-head"):
        x = x.to(device); y = y.to(device)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    torch.save(model.state_dict(), CLIP_CKPT)
    print("Saved CLIP detector weights:", CLIP_CKPT)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
/root/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 119.79it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.embeddings.position_ids                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight          

In [3]:
import torch

DELTA_PATH = os.path.join("paper_outputs", "universal_delta_bottom.pt")  # change if needed
assert os.path.exists(DELTA_PATH), f"Missing {DELTA_PATH}. Point to your saved delta .pt file."

delta_u_bottom = torch.load(DELTA_PATH, map_location=device)
if delta_u_bottom.ndim == 3:  # safety
    delta_u_bottom = delta_u_bottom.unsqueeze(0)

print("Loaded delta:", DELTA_PATH, "shape:", tuple(delta_u_bottom.shape))

Loaded delta: paper_outputs/universal_delta_bottom.pt shape: (1, 3, 224, 224)


In [8]:
# import torchvision.models as models
# import numpy as np
# from sklearn.metrics import roc_auc_score, accuracy_score
# from tqdm.auto import tqdm

# class ResNetDetector(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#         self.net.fc = nn.Linear(self.net.fc.in_features, 2)
#     def forward(self, x01):
#         return self.net(x01)

# resnet = ResNetDetector().to(device)
# opt_r = torch.optim.AdamW(resnet.parameters(), lr=3e-4, weight_decay=1e-4)

# @torch.no_grad()
# def eval_auc_acc(model_, loader_):
#      model_.eval()
#     ys, ps = [], []
#     for x, y, *_ in loader_:
#         x = x.to(device)
#         p = torch.softmax(model_(x), dim=-1)[:,1].detach().cpu().numpy()
#         ys.append(y.numpy()); ps.append(p)
#     y = np.concatenate(ys); p = np.concatenate(ps)
#     return {"auc": float(roc_auc_score(y, p)),
#             "acc@0.5": float(accuracy_score(y, (p >= 0.5).astype(int)))}

# # 2 epochs is usually enough
# for ep in range(2):
#     resnet.train()
#     for x, y, *_ in tqdm(train_loader_meta, desc=f"resnet train ep{ep+1}"):
#         x = x.to(device); y = y.to(device)
#         opt_r.zero_grad(set_to_none=True)
#         loss = F.cross_entropy(resnet(x), y)
#         loss.backward()
#         opt_r.step()

# print("ResNet CLEAN:", eval_auc_acc(resnet, test_loader_meta))
# torch.save(resnet.state_dict(), os.path.join("paper_outputs", "resnet_detector.pt"))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 246MB/s]
resnet train ep2: 100%|██████████| 25/25 [01:11<00:00,  2.85s/it]
ResNet CLEAN: {'auc': 0.99995, 'acc@0.5': 0.995}


In [11]:
# import pandas as pd
# import matplotlib.pyplot as plt

# def meme_band_mask(x, band_frac=0.22, where="bottom"):
#     B,C,H,W = x.shape
#     mask = torch.zeros_like(x)
#     h = int(H * band_frac)
#     if where == "bottom":
#         mask[:,:,H-h:H,:] = 1.0
#     elif where == "top":
#         mask[:,:,:h,:] = 1.0
#     else:
#         raise ValueError("where must be top/bottom")
#     return mask

# @torch.no_grad()
# def prob_fake_model(model_, x01):
#     p = torch.softmax(model_(x01), dim=-1)[:,1]
#     return torch.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)

# def eval_universal_transfer(model_, loader_, delta, mask_where="bottom", band_frac=0.22, tag=""):
#     rows = []
#     for x, y, pid, *_ in tqdm(loader_, desc=f"transfer-eval {tag}"):
#         x = x.to(device); y = y.to(device); pid = pid.to(device)

#         p0 = prob_fake_model(model_, x)

#         x_adv = x.clone()
#         fm = (y == 1)
#         if fm.any():
#             xf = x[fm]
#             m = meme_band_mask(xf, band_frac=band_frac, where=mask_where)
#             x_adv[fm] = (xf + delta.to(device) * m).clamp(0,1)

#         p1 = prob_fake_model(model_, x_adv)

#         for i in range(x.size(0)):
#             rows.append({
#                 "y": int(y[i].item()),
#                 "prompt_id": int(pid[i].item()),
#                 "p_fake_before": float(p0[i].item()),
#                 "p_fake_after": float(p1[i].item()),
#                 "model": tag,
#                 "mask_where": mask_where
#             })

#     df = pd.DataFrame(rows)
#     y_true = df["y"].values
#     p = df["p_fake_after"].values
#     auc = roc_auc_score(y_true, p)
#     acc = accuracy_score(y_true, (p >= 0.5).astype(int))

#     df_f = df[df["y"] == 1]
#     succ = float((df_f["p_fake_after"].values < 0.5).mean())

#     overall = {"model": tag, "mask_where": mask_where,
#                "auc_after": float(auc), "acc@0.5_after": float(acc),
#                "attack_success_fake_to_real@0.5": float(succ),
#                "n_fakes": int((df["y"]==1).sum()), "n_total": int(len(df))}
#     return df, overall

# # Evaluate transfer on CLIP and ResNet
# df_clip_tr, overall_clip_tr = eval_universal_transfer(model,  test_loader_meta, delta_u_bottom, "bottom", tag="CLIP")
# df_res_tr,  overall_res_tr  = eval_universal_transfer(resnet, test_loader_meta, delta_u_bottom, "bottom", tag="ResNet18")

# transfer_table = pd.DataFrame([overall_clip_tr, overall_res_tr])
# transfer_table.to_csv(os.path.join("paper_outputs", "transfer_universal_bottom.csv"), index=False)

# # Bar plot (paper figure)
# plt.figure()
# plt.bar(transfer_table["model"].values, transfer_table["attack_success_fake_to_real@0.5"].values)
# plt.ylabel("Universal attack success (fake→real @0.5)")
# plt.title("Cross-detector transfer of universal masked perturbation")
# plt.savefig(os.path.join("paper_outputs", "transfer_universal_bottom.png"), dpi=200, bbox_inches="tight")
# plt.close()

# transfer_table

transfer-eval ResNet18: 100%|██████████| 7/7 [00:25<00:00,  3.59s/it]


,model,mask_where,auc_after,acc@0.5_after,attack_success_fake_to_real@0.5,n_fakes,n_total
0,CLIP,bottom,0.996400,0.970,0.055,200,400
1,ResNet18,bottom,0.999975,0.995,0.005,200,400


In [4]:
# ============================================================
# ENSEMBLE UNIVERSAL MASKED EOT (CLIP + ResNet18) + PAPER OUTPUTS
# Goal: train ONE delta that fools BOTH detectors (fake -> real)
#
# Prereqs you should already have in memory:
#   - model      : CLIP detector (nn.Module)
#   - resnet     : ResNet18 detector (nn.Module)  (if not, load/train it first)
#   - train_loader_meta, test_loader_meta  (meta loaders)
#   - T          : FBLikeTransforms() (differentiable aug)
#   - meme_band_mask(x, band_frac, where)
#
# Saves into: paper_outputs/
#   - ensemble_delta_bottom_*.pt
#   - ensemble_universal_overall.csv
#   - ensemble_universal_per_sample_*.csv
#   - ensemble_universal_per_prompt_*.csv
#   - ensemble_universal_success_bar.png
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
OUTDIR = "paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)

@torch.no_grad()
def prob_fake_model(m, x01):
    p = torch.softmax(m(x01), dim=-1)[:, 1]
    return torch.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)

def apply_delta_masked(x01, delta, mask):
    return (x01 + delta * mask).clamp(0, 1)

print("Ready ✅")

Ready ✅


In [8]:
import torchvision.models as models

class ResNetDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.net.fc = nn.Linear(self.net.fc.in_features, 2)
    def forward(self, x01):
        return self.net(x01)

resnet = ResNetDetector().to(device)

RESNET_CKPT = "paper_outputs/resnet_detector.pt"
state_dict = torch.load(RESNET_CKPT, map_location=device)
resnet.load_state_dict(state_dict)

print("ResNet loaded correctly")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]
ResNet loaded correctly


In [17]:
!pip -q install -U kornia
import kornia.augmentation as K

# Optional differentiable JPEG; if install fails, EOT still works with geo+color+blur
try:
    !pip -q install -U diffjpeg
    from diffjpeg import DiffJPEG
    HAS_JPEG = True
except Exception:
    HAS_JPEG = False

class FBLikeTransforms(nn.Module):
    def __init__(self, h=224, w=224):
        super().__init__()
        self.geom = nn.Sequential(
            K.RandomResizedCrop((h,w), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=1.0),
        )
        self.color = nn.Sequential(
            K.RandomGamma(gamma=(0.9, 1.1), p=0.7),
            K.RandomBrightness(brightness=(0.9, 1.1), p=0.7),
            K.RandomContrast(contrast=(0.9, 1.1), p=0.7),
        )
        self.blur = K.RandomGaussianBlur((3,3), sigma=(0.1, 1.2), p=0.3)
        self.jpeg = DiffJPEG(height=h, width=w, differentiable=True, quality=85) if HAS_JPEG else None

    def forward(self, x01):
        x = self.geom(x01)
        x = self.color(x)
        x = self.blur(x)
        if self.jpeg is not None:
            x = self.jpeg(x.clamp(0,1))
        return x.clamp(0,1)

T = FBLikeTransforms().to(device)

def meme_band_mask(x, band_frac=0.22, where="bottom"):
    B,C,H,W = x.shape
    mask = torch.zeros_like(x)
    h = int(H * band_frac)
    if where == "bottom":
        mask[:,:,H-h:H,:] = 1.0
    elif where == "top":
        mask[:,:,:h,:] = 1.0
    else:
        raise ValueError("where must be top/bottom")
    return mask

def eot_pgd_targeted_masked(model, x01, y_target, T, eps=8/255, alpha=2/255, steps=30, eot_k=4, mask=None):
    """
    Attack only on masked region. Target label is 'real'(0) for fake samples.
    x01 in [0,1]
    """
    model.eval()
    x0 = x01.detach()
    if mask is None:
        mask = torch.ones_like(x0)

    delta = torch.zeros_like(x0).uniform_(-eps, eps).to(x0.device)
    delta = (delta * mask).requires_grad_(True)

    for _ in range(steps):
        loss = 0.0
        for _k in range(eot_k):
            adv = (x0 + delta).clamp(0,1)
            adv_t = T(adv)
            logits = model(adv_t)
            loss = loss + F.cross_entropy(logits, y_target)
        loss = loss / eot_k

        grad = torch.autograd.grad(loss, delta, retain_graph=False, create_graph=False)[0]
        delta.data = (delta.data - alpha * grad.sign()) * mask
        delta.data = delta.data.clamp(-eps, eps)
        delta.grad = None

    return (x0 + delta.detach()).clamp(0,1)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement diffjpeg (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for diffjpeg


In [20]:
# ============================================================
# (1) Ensemble universal training (stable PGD-style updates)
# Try these settings first on T4:
#   eps_u = 16/255   (also later run eps_u=8/255 for main)
#   step_u = 2/255
#   iters = 1200
#   eot_k = 4 (or 8 if time)
# ============================================================

def train_ensemble_universal_masked_eot(
    models,                 # list of detectors, e.g. [model, resnet]
    train_loader_meta,
    T,
    mask_where="bottom",
    band_frac=0.22,
    eps_u=16/255,
    step_u=2/255,
    iters=1200,
    eot_k=4,
    device="cuda"
):
    for m in models:
        m.eval()

    delta = torch.zeros(1, 3, 224, 224, device=device)

    loader_iter = iter(train_loader_meta)

    for it in tqdm(range(iters), desc=f"ensemble-universal-train-{mask_where}"):
        try:
            x, y, *_ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader_meta)
            x, y, *_ = next(loader_iter)

        x = x.to(device); y = y.to(device)
        fm = (y == 1)
        if fm.sum() == 0:
            continue
        xf = x[fm]

        msk = meme_band_mask(xf, band_frac=band_frac, where=mask_where)  # (B,3,H,W)
        m1 = msk[:1].detach()

        target = torch.zeros(xf.size(0), dtype=torch.long, device=device)  # real
        delta_var = delta.clone().detach().requires_grad_(True)

        loss = 0.0
        for _ in range(eot_k):
            adv = apply_delta_masked(xf, delta_var, msk)
            adv_t = T(adv)
            # sum losses over models (equal weight)
            for m in models:
                logits = m(adv_t)
                loss = loss + F.cross_entropy(logits, target)

        loss = loss / (eot_k * len(models))

        grad = torch.autograd.grad(loss, delta_var, retain_graph=False, create_graph=False)[0]
        grad = torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)

        with torch.no_grad():
            # targeted: descend loss
            delta = delta - step_u * grad.sign()
            delta = delta.clamp(-eps_u, eps_u)
            delta = delta * m1
            delta = torch.nan_to_num(delta, nan=0.0, posinf=0.0, neginf=0.0)

    return delta.detach()

# Train ensemble delta (bottom band)
delta_ens_bottom = train_ensemble_universal_masked_eot(
    models=[model, resnet],
    train_loader_meta=train_loader_meta,
    T=T,
    mask_where="bottom",
    band_frac=0.22,
    eps_u=16/255,
    step_u=2/255,
    iters=1200,
    eot_k=4,
    device=device
)

torch.save(delta_ens_bottom.cpu(), os.path.join(OUTDIR, "ensemble_delta_bottom_eps16.pt"))
print("Saved:", os.path.join(OUTDIR, "ensemble_delta_bottom_eps16.pt"))

ensemble-universal-train-bottom: 100%|██████████| 1200/1200 [53:45<00:00,  2.69s/it]
Saved: paper_outputs/ensemble_delta_bottom_eps16.pt


In [21]:
# ============================================================
# (2) Evaluate ensemble delta on BOTH models (with/without inference transforms)
# Produces paper-ready CSVs and a bar-plot
# ============================================================

def eval_universal_on_model(
    model_, loader_meta, delta, mask_where="bottom", band_frac=0.22,
    use_infer_T=False, tag=""
):
    model_.eval()
    rows = []

    for x, y, pid, is_fake, paths in tqdm(loader_meta, desc=f"eval-{tag}-{mask_where}-inferT{int(use_infer_T)}"):
        x = x.to(device); y = y.to(device); pid = pid.to(device)

        p0 = prob_fake_model(model_, x)

        x_adv = x.clone()
        fm = (y == 1)
        if fm.any():
            xf = x[fm]
            msk = meme_band_mask(xf, band_frac=band_frac, where=mask_where)
            x_adv[fm] = apply_delta_masked(xf, delta.to(device), msk)

        x_eval = T(x_adv) if use_infer_T else x_adv
        p1 = prob_fake_model(model_, x_eval)

        for i in range(x.size(0)):
            rows.append({
                "model": tag,
                "y": int(y[i].item()),
                "prompt_id": int(pid[i].item()),
                "p_fake_before": float(p0[i].item()),
                "p_fake_after": float(p1[i].item()),
                "mask_where": mask_where,
                "use_infer_T": int(use_infer_T),
                "path": paths[i]
            })

    df = pd.DataFrame(rows)

    # metrics
    y_true = df["y"].to_numpy().astype(int)
    p = df["p_fake_after"].to_numpy().astype(float)
    # sanitize just in case
    p = np.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)
    df["p_fake_after"] = p

    auc = roc_auc_score(y_true, p)
    acc = accuracy_score(y_true, (p >= 0.5).astype(int))

    df_f = df[df["y"] == 1]
    succ = float((df_f["p_fake_after"].to_numpy() < 0.5).mean())

    overall = {
        "model": tag,
        "mask_where": mask_where,
        "use_infer_T": int(use_infer_T),
        "auc_after": float(auc),
        "acc@0.5_after": float(acc),
        "attack_success_fake_to_real@0.5": float(succ),
        "n_total": int(len(df)),
        "n_fakes": int((df["y"] == 1).sum()),
    }

    # per-prompt on fakes
    per_prompt = []
    for pid_val, g in df[df["y"] == 1].groupby("prompt_id"):
        per_prompt.append({
            "model": tag,
            "use_infer_T": int(use_infer_T),
            "prompt_id": int(pid_val),
            "n": int(len(g)),
            "attack_success@0.5": float((g["p_fake_after"].to_numpy() < 0.5).mean()),
            "mean_p_fake_before": float(g["p_fake_before"].mean()),
            "mean_p_fake_after": float(g["p_fake_after"].mean()),
        })
    per_prompt_df = pd.DataFrame(per_prompt).sort_values(["model", "use_infer_T", "prompt_id"])

    return df, overall, per_prompt_df

# Evaluate on both models
results = []
pps = []
dfs = []

for m, name in [(model, "CLIP"), (resnet, "ResNet18")]:
    for inferT in [0, 1]:
        df_m, overall_m, pp_m = eval_universal_on_model(
            m, test_loader_meta, delta_ens_bottom,
            mask_where="bottom", band_frac=0.22,
            use_infer_T=bool(inferT),
            tag=name
        )
        results.append(overall_m)
        pps.append(pp_m)
        dfs.append(df_m)

overall_df = pd.DataFrame(results)
overall_df.to_csv(os.path.join(OUTDIR, "ensemble_universal_overall.csv"), index=False)

per_prompt_df = pd.concat(pps, ignore_index=True)
per_prompt_df.to_csv(os.path.join(OUTDIR, "ensemble_universal_per_prompt.csv"), index=False)

per_sample_df = pd.concat(dfs, ignore_index=True)
per_sample_df.to_csv(os.path.join(OUTDIR, "ensemble_universal_per_sample.csv"), index=False)

print("Saved:")
print(" -", os.path.join(OUTDIR, "ensemble_universal_overall.csv"))
print(" -", os.path.join(OUTDIR, "ensemble_universal_per_prompt.csv"))
print(" -", os.path.join(OUTDIR, "ensemble_universal_per_sample.csv"))
overall_df

eval-ResNet18-bottom-inferT1: 100%|██████████| 7/7 [00:26<00:00,  3.79s/it]
Saved:
 - paper_outputs/ensemble_universal_overall.csv
 - paper_outputs/ensemble_universal_per_prompt.csv
 - paper_outputs/ensemble_universal_per_sample.csv


,model,mask_where,use_infer_T,auc_after,acc@0.5_after,attack_success_fake_to_real@0.5,n_total,n_fakes
0,CLIP,bottom,0,0.922925,0.825,0.345,400,200
1,CLIP,bottom,1,0.938625,0.820,0.355,400,200
2,ResNet18,bottom,0,0.996000,0.945,0.105,400,200
3,ResNet18,bottom,1,0.969875,0.845,0.310,400,200


In [13]:
# ============================================================
# (3) Paper figure: bar chart of attack success across models (+inferT)

import pandas as pd
import matplotlib.pyplot as plt
# ============================================================
OUTDIR="paper_outputs"
overall_df=pd.read_csv(os.path.join(OUTDIR, "ensemble_universal_overall.csv"))
plot_df = overall_df.copy()
plot_df["label"] = plot_df["model"] + plot_df["use_infer_T"].map({0:" (no T)", 1:" (+T)"})

plt.figure()
plt.bar(plot_df["label"].values, plot_df["attack_success_fake_to_real@0.5"].values)
plt.ylabel("Universal attack success (fake→real @0.5)")
plt.title("Ensemble-trained universal masked perturbation (transfer by design)")
plt.xticks(rotation=20, ha="right")
plt.savefig(os.path.join(OUTDIR, "ensemble_universal_success_bar.png"), dpi=200, bbox_inches="tight")
plt.close()

print("Saved figure:", os.path.join(OUTDIR, "ensemble_universal_success_bar.png"))

Saved figure: paper_outputs/ensemble_universal_success_bar.png


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=dbc8aaad-7458-4d3a-af75-0f37995653fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>